# Ejercicios — Vibe Coding: LLMs, Tools y Agentes

10 ejercicios para practicar los patrones que una IA usa al generar codigo Python.

## Estructura

| Tier | Ejercicios | Cuando |
|---|---|---|
| **Nucleo** | 1–5 | Obligatorio en clase |
| **Si vamos bien** | 6–7 | Si el ritmo lo permite |
| **Bonus / casa** | 8–10 | Tarea o alumnos adelantados |

Cada ejercicio tiene: enunciado → tu codigo → validador → solucion guiada.

In [ ]:
# Clases base que necesitaras en los ejercicios (ejecuta esta celda primero)

class Trade:
    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        self.symbol = symbol
        self.side = side
        self.price = price
        self.size = size

    def cash_flow(self) -> float:
        signed = -1 if self.side == "buy" else 1
        return signed * self.price * self.size

class Order:
    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        self.symbol = symbol
        self.side = side
        self.price = price
        self.size = size

    def notional(self) -> float:
        return self.price * self.size

    def describe(self) -> str:
        return f"{self.side} {self.size:.2f} {self.symbol} @ {self.price}"

    def __repr__(self) -> str:
        return f"Order({self.symbol}, {self.side}, price={self.price})"

print("Trade y Order cargados correctamente.")

---
## Ejercicio 1 — `try/except` basico

Escribe una funcion `safe_notional(price, size)` que:
- devuelva `price * size`
- si `price` o `size` no son numeros (TypeError), devuelva `0.0`
- si `size` es cero y se intenta dividir (no aplica aqui, pero practica el patron), devuelva `0.0`

Usa type hints: `def safe_notional(price: float, size: float) -> float`

In [ ]:
def safe_notional(price: float, size: float) -> float:
    pass  # tu codigo aqui

In [ ]:
# --- Validador Ejercicio 1 ---
assert safe_notional(100000, 0.10) == 10000.0, f"expected 10000.0, got {safe_notional(100000, 0.10)}"
assert safe_notional("abc", 0.10) == 0.0, "should return 0.0 for non-numeric price"
assert safe_notional(100000, "x") == 0.0, "should return 0.0 for non-numeric size"
assert safe_notional(0, 0.10) == 0.0, f"expected 0.0 for price=0, got {safe_notional(0, 0.10)}"
print("Ejercicio 1 OK")

In [ ]:
# --- Solucion guiada Ejercicio 1 ---
def safe_notional(price: float, size: float) -> float:
    try:
        result = price * size
        return float(result)
    except (TypeError, ValueError):
        return 0.0

print("safe_notional(100000, 0.10):", safe_notional(100000, 0.10))
print("safe_notional('abc', 0.10):", safe_notional("abc", 0.10))

---
## Ejercicio 2 — Type hints

Escribe una funcion `classify_order` con esta firma:

```python
def classify_order(price: float, size: float, threshold: float = 5000.0) -> str:
```

Debe devolver:
- `"large"` si `price * size >= threshold`
- `"small"` en caso contrario

In [ ]:
def classify_order(price: float, size: float, threshold: float = 5000.0) -> str:
    pass  # tu codigo aqui

In [ ]:
# --- Validador Ejercicio 2 ---
assert classify_order(100000, 0.10) == "large", f"expected 'large', got {classify_order(100000, 0.10)}"
assert classify_order(100, 0.01) == "small", f"expected 'small', got {classify_order(100, 0.01)}"
assert classify_order(5000, 1.0) == "large", "5000 * 1.0 = 5000 >= threshold"
assert classify_order(100000, 0.10, threshold=20000.0) == "small", "custom threshold should work"
# Check type hints exist
import inspect
sig = inspect.signature(classify_order)
assert sig.return_annotation == str, "return type hint should be str"
print("Ejercicio 2 OK")

In [ ]:
# --- Solucion guiada Ejercicio 2 ---
def classify_order(price: float, size: float, threshold: float = 5000.0) -> str:
    notional = price * size
    return "large" if notional >= threshold else "small"

print(classify_order(100000, 0.10))          # large
print(classify_order(100, 0.01))             # small
print(classify_order(100000, 0.10, 20000))   # small (custom threshold)

---
## Ejercicio 3 — `@property`

Crea una clase `SafeOrder` con:
- `__init__(self, symbol: str, side: str, price: float, size: float)`
- Atributos internos con `_` prefix (`_symbol`, `_side`, `_price`, `_size`)
- `@property` para `symbol`, `side`, `price`, `size` (solo lectura)
- `@property notional` que devuelva `price * size`

In [ ]:
class SafeOrder:
    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        pass  # tu codigo aqui

In [ ]:
# --- Validador Ejercicio 3 ---
so = SafeOrder("BTCUSDT", "buy", 100000, 0.10)
assert so.symbol == "BTCUSDT", f"expected BTCUSDT, got {so.symbol}"
assert so.side == "buy", f"expected buy, got {so.side}"
assert so.price == 100000, f"expected 100000, got {so.price}"
assert so.size == 0.10, f"expected 0.10, got {so.size}"
assert abs(so.notional - 10000.0) < 1e-9, f"expected notional=10000.0, got {so.notional}"
# Check read-only
try:
    so.price = 999
    assert False, "price should be read-only (no setter)"
except AttributeError:
    pass
print("Ejercicio 3 OK")

In [ ]:
# --- Solucion guiada Ejercicio 3 ---
class SafeOrder:
    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        self._symbol = symbol
        self._side = side
        self._price = price
        self._size = size

    @property
    def symbol(self) -> str:
        return self._symbol

    @property
    def side(self) -> str:
        return self._side

    @property
    def price(self) -> float:
        return self._price

    @property
    def size(self) -> float:
        return self._size

    @property
    def notional(self) -> float:
        return self._price * self._size

so = SafeOrder("BTCUSDT", "buy", 100000, 0.10)
print(f"{so.symbol} {so.side} notional={so.notional}")

---
## Ejercicio 4 — List comprehension

Dada la lista `orders`, crea:
- `notionals`: lista con el notional de cada orden (comprehension)
- `big_buys`: lista de ordenes donde `side == "buy"` y `notional >= 5000` (comprehension con filtro)

In [ ]:
orders = [
    {"symbol": "BTCUSDT", "side": "buy",  "price": 100000, "size": 0.10},
    {"symbol": "BTCUSDT", "side": "sell", "price": 100020, "size": 0.08},
    {"symbol": "ETHUSDT", "side": "buy",  "price": 3520,   "size": 1.40},
    {"symbol": "ETHUSDT", "side": "sell", "price": 3530,   "size": 0.50},
    {"symbol": "BTCUSDT", "side": "buy",  "price": 99980,  "size": 0.05},
]

notionals = None  # tu comprehension aqui
big_buys = None   # tu comprehension aqui

In [ ]:
# --- Validador Ejercicio 4 ---
assert isinstance(notionals, list), "notionals debe ser una lista"
assert len(notionals) == 5, f"notionals debe tener 5 elementos, tiene {len(notionals)}"
assert abs(notionals[0] - 10000.0) < 1e-9, f"primer notional deberia ser 10000.0, es {notionals[0]}"
assert abs(notionals[2] - 4928.0) < 1e-9, f"tercer notional deberia ser 4928.0, es {notionals[2]}"
assert isinstance(big_buys, list), "big_buys debe ser una lista"
assert len(big_buys) == 2, f"big_buys debe tener 2 elementos, tiene {len(big_buys)}"
assert big_buys[0]["symbol"] == "BTCUSDT", "primer big buy deberia ser BTCUSDT"
assert big_buys[1]["symbol"] == "ETHUSDT", "segundo big buy deberia ser ETHUSDT (notional 4928)"
print("Ejercicio 4 OK")

In [ ]:
# --- Solucion guiada Ejercicio 4 ---
notionals = [o["price"] * o["size"] for o in orders]
big_buys = [o for o in orders if o["side"] == "buy" and o["price"] * o["size"] >= 5000]

print("notionals:", notionals)
print("big_buys:", big_buys)

---
## Ejercicio 5 — Evaluar codigo de IA

Una IA genero esta clase `Order`. Comparala con la version de Lesson 2. Encuentra **2 diferencias** y explica si mejoran o no el codigo.

Escribe tus respuestas en las variables `diferencia_1` y `diferencia_2` como strings.

In [ ]:
# --- Version L2 (la que escribiste a mano) ---
# class Order:
#     def __init__(self, symbol, side, price, size):
#         self.symbol = symbol
#         self.side = side
#         self.price = price
#         self.size = size
#
#     def notional(self):
#         return self.price * self.size

# --- Version IA ---
class OrderAI:
    """Represents a trading order with validation."""

    VALID_SIDES = ("buy", "sell")

    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        if side not in self.VALID_SIDES:
            raise ValueError(f"Invalid side: {side}")
        self._symbol = symbol
        self._side = side
        self._price = price
        self._size = size

    @property
    def notional(self) -> float:
        return self._price * self._size

    def describe(self) -> str:
        return f"{self._side} {self._size:.2f} {self._symbol} @ {self._price:,.0f}"


# ¿Que diferencias ves? Escribe al menos 2.
diferencia_1 = None  # ej: "La version IA usa type hints en __init__"
diferencia_2 = None  # ej: "La version IA valida side en __init__"

In [ ]:
# --- Validador Ejercicio 5 ---
assert diferencia_1 is not None and len(diferencia_1) > 10, "diferencia_1 debe ser un string descriptivo"
assert diferencia_2 is not None and len(diferencia_2) > 10, "diferencia_2 debe ser un string descriptivo"
assert isinstance(diferencia_1, str), "diferencia_1 debe ser un string"
assert isinstance(diferencia_2, str), "diferencia_2 debe ser un string"
print("Ejercicio 5 OK — tus respuestas:")
print(f"  1: {diferencia_1}")
print(f"  2: {diferencia_2}")

In [ ]:
# --- Solucion guiada Ejercicio 5 ---
# Diferencias principales entre la version L2 y la version IA:
#
# 1. Type hints: la IA aniade `: str`, `: float`, `-> float` a todos los metodos.
#    Mejora: si, porque documenta la interfaz sin necesidad de leer el cuerpo.
#
# 2. Validacion en __init__: la IA valida que side sea "buy" o "sell".
#    Mejora: si, porque previene errores silenciosos (ej: side="hold" no crashea en L2).
#
# 3. @property para notional: en L2 es un metodo (order.notional()), en la IA es
#    una propiedad (order.notional sin parentesis). Discutible: ambos son validos.
#
# 4. Atributos privados (_symbol en vez de symbol): la IA usa convencion de
#    encapsulacion mas estricta. Mejora: si, combinado con @property da read-only.
#
# 5. Docstring y VALID_SIDES como constante de clase.

diferencia_1 = "La version IA usa type hints en __init__ y en todos los metodos"
diferencia_2 = "La version IA valida side en __init__ con raise ValueError"
print("Respuestas de ejemplo guardadas.")

---
## Ejercicio 6 — Decorador simple

Crea un decorador `validate_positive` que envuelva una funcion y lance `ValueError` si algun argumento posicional es negativo. Aplica el decorador a una funcion `compute_notional(price, size)`.

In [ ]:
def validate_positive(func):
    pass  # tu codigo aqui — define wrapper, comprueba args, devuelve wrapper


@validate_positive
def compute_notional(price: float, size: float) -> float:
    return price * size

In [ ]:
# --- Validador Ejercicio 6 ---
assert compute_notional(100000, 0.10) == 10000.0, f"expected 10000.0, got {compute_notional(100000, 0.10)}"
try:
    compute_notional(-100, 0.10)
    assert False, "should raise ValueError for negative price"
except ValueError:
    pass
try:
    compute_notional(100000, -0.05)
    assert False, "should raise ValueError for negative size"
except ValueError:
    pass
print("Ejercicio 6 OK")

In [ ]:
# --- Solucion guiada Ejercicio 6 ---
def validate_positive(func):
    def wrapper(*args, **kwargs):
        for i, arg in enumerate(args):
            if isinstance(arg, (int, float)) and arg < 0:
                raise ValueError(f"argument {i} is negative: {arg}")
        return func(*args, **kwargs)
    return wrapper

@validate_positive
def compute_notional(price: float, size: float) -> float:
    return price * size

print(compute_notional(100000, 0.10))  # 10000.0
try:
    print(compute_notional(-100, 0.10))
except ValueError as e:
    print(f"Caught: {e}")

---
## Ejercicio 7 — Dict comprehension

Dada la lista `orders`, crea un diccionario `volume_by_symbol` donde:
- la clave sea el `symbol`
- el valor sea la suma de `size` de todas las ordenes de ese symbol

Pista: puedes usar un set comprehension para obtener los symbols unicos primero, y luego un dict comprehension.

In [ ]:
orders = [
    {"symbol": "BTCUSDT", "side": "buy",  "price": 100000, "size": 0.10},
    {"symbol": "BTCUSDT", "side": "sell", "price": 100020, "size": 0.08},
    {"symbol": "ETHUSDT", "side": "buy",  "price": 3520,   "size": 1.40},
    {"symbol": "ETHUSDT", "side": "sell", "price": 3530,   "size": 0.50},
    {"symbol": "BTCUSDT", "side": "buy",  "price": 99980,  "size": 0.05},
]

volume_by_symbol = None  # tu dict comprehension aqui

In [ ]:
# --- Validador Ejercicio 7 ---
assert isinstance(volume_by_symbol, dict), "volume_by_symbol debe ser un dict"
assert len(volume_by_symbol) == 2, f"debe tener 2 symbols, tiene {len(volume_by_symbol)}"
assert abs(volume_by_symbol["BTCUSDT"] - 0.23) < 1e-9, f"BTCUSDT volume debe ser 0.23, es {volume_by_symbol['BTCUSDT']}"
assert abs(volume_by_symbol["ETHUSDT"] - 1.90) < 1e-9, f"ETHUSDT volume debe ser 1.90, es {volume_by_symbol['ETHUSDT']}"
print("Ejercicio 7 OK")
print("volume_by_symbol:", volume_by_symbol)

In [ ]:
# --- Solucion guiada Ejercicio 7 ---
symbols = {o["symbol"] for o in orders}  # set comprehension
volume_by_symbol = {
    s: sum(o["size"] for o in orders if o["symbol"] == s)
    for s in symbols
}
print("volume_by_symbol:", volume_by_symbol)

---
## Ejercicio 8 — Predecir output de IA

Lee el siguiente codigo (simulando output de una IA) y **sin ejecutarlo** predice:
1. ¿Que imprime `tracker.status`?
2. ¿Hay algun bug?

Escribe tus predicciones en `prediction_status` y `prediction_bug`. Despues ejecuta para verificar.

In [ ]:
# --- Codigo generado por IA (leelo antes de ejecutar) ---
class MiniTracker:
    def __init__(self) -> None:
        self._cash: float = 0.0
        self._position: float = 0.0
        self._trade_count: int = 0

    def apply(self, side: str, price: float, size: float) -> None:
        if side == "buy":
            self._cash -= price * size
            self._position += size
        elif side == "sell":
            self._cash += price * size
            self._position -= size
        self._trade_count += 1

    @property
    def status(self) -> str:
        if self._position > 0:
            return "long"
        elif self._position < 0:
            return "short"
        return "flat"

    @property
    def pnl(self) -> float:
        return self._cash  # BUG: ignora position no cerrada


# Escribe tus predicciones ANTES de ejecutar
prediction_status = None   # ¿que sera tracker.status despues de las 3 operaciones?
prediction_bug = None      # ¿que esta mal en la propiedad pnl?

# Operaciones
tracker = MiniTracker()
tracker.apply("buy", 100000, 0.10)
tracker.apply("buy", 99900, 0.05)
tracker.apply("sell", 100200, 0.08)

In [ ]:
# --- Validador Ejercicio 8 ---
assert prediction_status is not None, "escribe tu prediccion en prediction_status"
assert prediction_bug is not None, "escribe tu prediccion en prediction_bug"
print(f"Tu prediccion de status: {prediction_status}")
print(f"Status real: {tracker.status}")
print(f"Position: {tracker._position} (buy 0.10 + buy 0.05 - sell 0.08 = 0.07 → long)")
print(f"\nTu prediccion de bug: {prediction_bug}")
print(f"pnl reportado: {tracker.pnl:.2f}")
print(f"pnl real deberia considerar: cash + position * mark_price")
print("Ejercicio 8 OK")

In [ ]:
# --- Solucion guiada Ejercicio 8 ---
# Status: "long" (position = 0.10 + 0.05 - 0.08 = 0.07 > 0)
# Bug: pnl solo devuelve cash, pero si tienes position abierta (0.07 BTC),
#      el pnl real depende del mark price. Deberia ser:
#      def pnl(self, mark_price): return self._cash + self._position * mark_price
#      O al menos equity() en vez de pnl.
prediction_status = "long"
prediction_bug = "pnl ignora la posicion abierta — deberia sumar position * mark_price"
print("Predicciones de ejemplo guardadas.")

---
## Ejercicio 9 — Refactorizar L2 con patrones nuevos

Toma la clase `Order` de Lesson 2 y refactorizala con todo lo que has aprendido hoy:

- Atributos privados con `_` prefix
- `@property` para cada atributo (read-only)
- `@property notional` (sin parentesis)
- Type hints en todos los metodos
- Validacion en `__init__` que lance `ValueError` si `price <= 0` o `size <= 0`

Llamala `OrderV3`.

In [ ]:
class OrderV3:
    pass  # tu codigo aqui

In [ ]:
# --- Validador Ejercicio 9 ---
o3 = OrderV3("BTCUSDT", "buy", 100000, 0.10)
assert o3.symbol == "BTCUSDT", f"symbol: expected BTCUSDT, got {o3.symbol}"
assert o3.side == "buy"
assert o3.price == 100000
assert o3.size == 0.10
assert abs(o3.notional - 10000.0) < 1e-9, f"notional: expected 10000.0, got {o3.notional}"
# Check read-only
try:
    o3.symbol = "ETHUSDT"
    assert False, "symbol should be read-only"
except AttributeError:
    pass
# Check validation
try:
    OrderV3("BTCUSDT", "buy", -100, 0.10)
    assert False, "should reject negative price"
except ValueError:
    pass
try:
    OrderV3("BTCUSDT", "buy", 100000, 0)
    assert False, "should reject zero size"
except ValueError:
    pass
print("Ejercicio 9 OK")

In [ ]:
# --- Solucion guiada Ejercicio 9 ---
class OrderV3:
    def __init__(self, symbol: str, side: str, price: float, size: float) -> None:
        if price <= 0:
            raise ValueError(f"price must be positive, got {price}")
        if size <= 0:
            raise ValueError(f"size must be positive, got {size}")
        self._symbol = symbol
        self._side = side
        self._price = price
        self._size = size

    @property
    def symbol(self) -> str:
        return self._symbol

    @property
    def side(self) -> str:
        return self._side

    @property
    def price(self) -> float:
        return self._price

    @property
    def size(self) -> float:
        return self._size

    @property
    def notional(self) -> float:
        return self._price * self._size

    def describe(self) -> str:
        return f"{self._side} {self._size:.2f} {self._symbol} @ {self._price:,.0f}"

    def __repr__(self) -> str:
        return f"OrderV3({self._symbol}, {self._side}, {self._price}, {self._size})"

o3 = OrderV3("BTCUSDT", "buy", 100000, 0.10)
print(o3.describe())
print(f"notional: {o3.notional}")
print(repr(o3))

---
## Ejercicio 10 — Genera un `TradeLogger`

Imagina que le pides a un LLM: *"Crea una clase TradeLogger que registre trades y pueda devolver un resumen."*

Ahora **tu** eres el LLM. Genera la clase `TradeLogger` con:

- `__init__` que inicialice una lista interna `_trades`
- `log(self, trade: Trade) -> None` que anada el trade a la lista
- `@property trade_count -> int`
- `@property total_volume -> float` (suma de `size` de todos los trades)
- `summary(self) -> str` con formato: `"N trades | volume: X.XXXX"`
- `try/except` en `total_volume` por si la lista tiene datos corruptos

In [ ]:
class TradeLogger:
    pass  # tu codigo aqui — genera la clase completa

In [ ]:
# --- Validador Ejercicio 10 ---
logger = TradeLogger()
logger.log(Trade("BTCUSDT", "buy", 100000, 0.10))
logger.log(Trade("BTCUSDT", "sell", 100020, 0.08))
logger.log(Trade("ETHUSDT", "buy", 3520, 1.40))

assert logger.trade_count == 3, f"trade_count: expected 3, got {logger.trade_count}"
assert abs(logger.total_volume - 1.58) < 1e-9, f"total_volume: expected 1.58, got {logger.total_volume}"
s = logger.summary()
assert "3" in s, "summary should contain trade count"
assert isinstance(s, str), "summary should return a string"
# Check empty logger
empty_logger = TradeLogger()
assert empty_logger.trade_count == 0
assert empty_logger.total_volume == 0.0
print("Ejercicio 10 OK")
print("summary:", logger.summary())

In [ ]:
# --- Solucion guiada Ejercicio 10 ---
class TradeLogger:
    def __init__(self) -> None:
        self._trades: list[Trade] = []

    def log(self, trade: Trade) -> None:
        self._trades.append(trade)

    @property
    def trade_count(self) -> int:
        return len(self._trades)

    @property
    def total_volume(self) -> float:
        try:
            return sum(t.size for t in self._trades)
        except (AttributeError, TypeError):
            return 0.0

    def summary(self) -> str:
        return f"{self.trade_count} trades | volume: {self.total_volume:.4f}"

logger = TradeLogger()
logger.log(Trade("BTCUSDT", "buy", 100000, 0.10))
logger.log(Trade("BTCUSDT", "sell", 100020, 0.08))
logger.log(Trade("ETHUSDT", "buy", 3520, 1.40))
print(logger.summary())

---
## Cierre

Has practicado los 5 patrones que un LLM usa al generar codigo Python:

| Patron | Que hace | Ejercicios |
|---|---|---|
| `try/except` | Captura errores sin crashear | 1, 8, 10 |
| Type hints | Documenta la interfaz en la firma | 2, 5, 9 |
| `@property` | Acceso controlado a estado interno | 3, 8, 9, 10 |
| Comprehensions | For+append en una linea | 4, 7 |
| Decoradores | Envolver funciones con comportamiento extra | 6 |

**Siguiente clase:** ya sabes Python, OOP y como trabajar con IA. Toca datos reales: microestructura de mercado con BTC.

La IA genera. Tu evaluas. El mercado no perdona errores.